# 2.2. 数据预处理
在Python中常用的数据分析工具中，我们通常使用pandas软件包。 像庞大的Python生态系统中的许多其他扩展包一样，pandas可以与张量兼容。

### 2.2.1 读取数据集

In [3]:
# 1. 首先创建一个人工数据集，并存储在csv文件中（下面将数据集按行写入CSV文件中）
import os

os.makedirs(os.path.join('..','data'), exist_ok=True)
data_file = os.path.join('..', 'data', 'house_tiny.csv')
with open(data_file, 'w') as f:
    f.write('NumRooms,Alley,Price\n')  # 列名
    f.write('NA,Pave,127500\n')        # 每行表示一个样本
    f.write('2,NA,106000\n')
    f.write('4,NA,178100\n')
    f.write('NA,NA,140000\n')

In [4]:
# 2. 使用pandas包中的read_csv函数来从csv文件中加载原始数据集
import pandas as pd

data = pd.read_csv(data_file)
print(data)

   NumRooms Alley   Price
0       NaN  Pave  127500
1       2.0   NaN  106000
2       4.0   NaN  178100
3       NaN   NaN  140000


### 2.2.2 处理缺失值

注意，“NaN”项代表缺失值。 为了处理缺失的数据，典型的方法包括插值法和删除法， 其中插值法用一个替代值弥补缺失值，而删除法则直接忽略缺失值。

In [5]:
# 1. 使用插值法处理缺失值（这里采用同列的均值来替换掉NaN）
inputs = data.iloc[:,0:2]
outputs = data.iloc[:,2]
inputs = inputs.fillna(inputs.mean(numeric_only=True)) # 传入 numeric_only=True 参数，让mean函数只计算数值列的均值
print(inputs)

   NumRooms Alley
0       3.0  Pave
1       2.0   NaN
2       4.0   NaN
3       3.0   NaN


对于inputs中的类别值或离散值，我们将“NaN”视为一个类别。 由于“巷子类型”（“Alley”）列只接受两种类型的类别值“Pave”和“NaN”， pandas可以自动将此列转换为两列“Alley_Pave”和“Alley_nan”。 巷子类型为“Pave”的行会将“Alley_Pave”的值设置为1，“Alley_nan”的值设置为0。 缺少巷子类型的行会将“Alley_Pave”和“Alley_nan”分别设置为0和1。<br>
pd.get_dummies(inputs, dummy_na=True) 运行后的变化：<br>
这个函数看到 Alley 这一列是文本，就把它拆开成了两列，分别叫 Alley_Pave 和 Alley_nan（注意这里 dummy_na=True 的作用就是把缺失值 NaN 也当成一种正常的类别）。
- 对于第1行：你是 Pave 吗？是(1)。 你是 NaN 吗？否(0)。
- 对于第2行：你是 Pave 吗？否(0)。 你是 NaN 吗？是(1)。

In [8]:
inputs = pd.get_dummies(inputs, dummy_na=True) # 将文本类别转换成机器能看懂的数字（one-hot编码）
print(inputs)

   NumRooms  Alley_Pave  Alley_nan
0       3.0        True      False
1       2.0       False       True
2       4.0       False       True
3       3.0       False       True


### 2.2.3. 转换为张量格式
经过上边的处理后，inputs和outputs中都是数值类型，因此可以转换成张量格式

In [9]:
import torch 
X = torch.tensor(inputs.to_numpy(dtype=float))
y = torch.tensor(outputs.to_numpy(dtype=float))
X, y

(tensor([[3., 1., 0.],
         [2., 0., 1.],
         [4., 0., 1.],
         [3., 0., 1.]], dtype=torch.float64),
 tensor([127500., 106000., 178100., 140000.], dtype=torch.float64))

### 2.2.4. 小结
- pandas软件包是Python中常用的数据分析工具中，pandas可以与张量兼容。

- 用pandas处理缺失的数据时，我们可根据情况选择用插值法和删除法。

### 2.2.5. 练习

In [13]:
# 1. 创建包含更多行和更多列的原始数据集
data_file = os.path.join('..', 'data', 'score.csv')
with open(data_file, 'w') as f:
    f.write('Total,Math,English,Chinese,Chemistry,Physics\n')
    f.write('500,100,na,100,100,100\n')
    f.write('400,na,95,60,70,80\n')
    f.write('450,67,78,23,100,35\n')
    f.write('380,47,na,67,34,81\n')

In [ ]:
import pandas as pd

# 读取原始数据集（如果有小写的 'na'，让 pandas 识别为真正的缺失值 NaN）
data = pd.read_csv(data_file, na_values=['na'])
print("【原始数据】:")
print(data)

def drop_col_with_most_na(df):
    """
    统计每一列缺失值的个数，并删除缺失值最多的那列
    """
    # 1. 统计每一列缺失值(NaN)的个数
    na_counts = df.isna().sum()
    print("\n【各列缺失值统计】:")
    print(na_counts)
    
    # 2. 找到缺失值最多的列名
    max_na_col = na_counts.idxmax()
    print(f"\n -> 找出缺失值最多的列: '{max_na_col}'，执行删除。")
    
    # 3. 删除这列并返回
    return df.drop(columns=max_na_col)

# 调用写好的函数
data_updated = drop_col_with_most_na(data)

print("\n【删除后的结果】:")
print(data_updated)

【原始数据】:
   Total Math English  Chinese  Chemistry  Physics
0    500  100      na      100        100      100
1    400   na      95       60         70       80
2    450   67      78       23        100       35
3    380   47      na       67         34       81
